<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string
import re

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [2]:
! kaggle datasets download -d gsimonx37/letterboxd

Dataset URL: https://www.kaggle.com/datasets/gsimonx37/letterboxd
License(s): GPL-3.0
100% 22.5G/22.5G [02:42<00:00, 141MB/s]
100% 22.5G/22.5G [02:42<00:00, 149MB/s]


We only consider a subet of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [3]:
import zipfile
from multiprocessing import Pool

DATA_DIR = "./letterboxd"
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
    for file_name in members_to_extract:
        zip_ref.extract(file_name + '.csv', DATA_DIR)


We then prepare the entry point for the Spark functionalities that will we use from now on.

In [4]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!rm spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form.

In [39]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

def extract_data(member):
    rdd = sc.textFile(DATA_DIR + "/" + member + ".csv")

    if member == 'actors':
        rdd = rdd.zipWithIndex().map(lambda r: r[0] + ',' + str(r[1]))
        rdd = rdd.map(lambda r: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', r))
        problem = rdd.filter(lambda r: '1047317' in r).collect()
        print(problem)
    rdd = rdd.map(lambda r: re.split(r',(?! )', r))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions

    # get column name from csv different from 'id'
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    if member == 'actors':
        column_names[-1] = 'movie_n'
    rdd = (rdd
            .map(lambda r: (r[0], dict(zip(column_names, r[1:]))))
            .filter(lambda r: r[0]!='id'))
    return rdd

for member in members_to_extract:
    letterboxd_RDDs[member] = extract_data(member)

['1047317,Lupita Tovar,Santa,1027990', '1047317,Carlos Orellana,Hipólito,1027991', '1047317,Juan José Martínez Casado,Jarameño,1027992', '1047317,Donald Reed,Marcelino,1027993', '1047317,Antonio R. Frausto,Fabián,1027994', '1047317,Mimí Derba,Doña Elvira,1027995', "1047317,Rosita Arriaga,Santa's Mother,1027996", '1047317,Joaquín Busquets,Esteban,1027997', '1047317,Feliciano Rueda,Drunk at brothel,1027998', '1047317,Jorge Peón,Genarillo,1027999', "1047317,Alberto Martí,Jarameño's friend,1028000", '1047317,Ricardo Carti,Doctor,1028001', '1047317,Sofía Álvarez,Prostitute 1,1028002', '1047317,Rosa Castro,Prostitute 2,1028003', '1047317,Nena Betancourt,Singer,1028004', '1047317,Jorge Marrón,,1028005', '1047317,Carlos Bocanegra,,1028006', '1047317,Fernando A. Rivero,,1028007', '1047317,Lupita Gallardo,Prostitute 3,1028008', '1047317,Ismael Rodríguez,,1028009', '1047317,Raúl de Anda,,1028010', '1047317,Parkey Hussian,,1028011', '1047317,Cube Bonifant,,1028012', '1048670,Maxine Denis,Dalia,104

Let's look at the amount of rows for each data type.

In [18]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	5798450
Number of rows for crew:	4720183
Number of rows for genres:	1046849
Number of rows for movies:	941597
Number of rows for themes:	125641


A glimpse at the structure of the rows in the RDDs of each data type.

In [19]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 ('1000001', {'name': 'Margot Robbie', 'role': 'Barbie', 'movie_n': '1'})
Row for crew:	 ('1000001', {'role': 'Director', 'name': 'Greta Gerwig'})
Row for genres:	 ('1000001', {'genre': 'Comedy'})
Row for movies:	 ('1000001', {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
Row for themes:	 ('1000001', {'theme': 'Humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.

In [41]:
def get_sample(rdd, size):
    return rdd.filter(lambda r: int(r[0],10)<=1000000+size)

for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], 100)

In [42]:
print(letterboxd_RDDs['actors'].count())
print(letterboxd_RDDs['crew'].count())
print(letterboxd_RDDs['genres'].count())
print(letterboxd_RDDs['movies'].count())
print(letterboxd_RDDs['themes'].count())

6075
8943
272
99
696


In [ ]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 45.0 failed 1 times, most recent failure: Lost task 1.0 in stage 45.0 (TID 163) (6ea7ff692cfd executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/content/spark-3.5.3-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/content/spark-3.5.3-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 1237, in process
    out_iter = func(split_index, iterator)
  File "/content/spark-3.5.3-bin-hadoop3/python/pyspark/rdd.py", line 5434, in pipeline_func
    return func(split, prev_func(split, iterator))
  File "/content/spark-3.5.3-bin-hadoop3/python/pyspark/rdd.py", line 5434, in pipeline_func
    return func(split, prev_func(split, iterator))
  File "/content/spark-3.5.3-bin-hadoop3/python/pyspark/rdd.py", 

For each data type the available attributes are the following:

| **Table**    | **Columns**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genre                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | theme                               |

For this project we would like to focus on the following features:

| **Table**    | **Columns**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genre                               |
| **movies**   | name, date, minute, rating |
| **themes**   | theme                               |


We keep only the 10 most relevant actors in each movie.

In [ ]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], sorted(r[1], key=lambda x: x["movie_n"])[:10])))

In [ ]:

def filter_dict_fields(d, final_fields):
    return {key: d[key] for key in final_fields if key in d}

def rename_key_in_dict(d, old_key, new_key):
    d[new_key] = d.pop(old_key)
    return d

In [ ]:
rdd_1 = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], {key: [d[key] for d in r[1]] for key in r[1][0]}))
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'actors'))))

We filter only the directors from the crew dataset, and we ony keep their name.

In [ ]:
letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                           .filter(lambda r: r[1]['role']=='Director')
                           .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                           .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'director'))))

For each movie we only store the attributes listed in the table above.

In [ ]:
letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name', 'date', 'minute', 'rating']))))

In [ ]:
rdd_1.take(10)

In [ ]:
for member in members_to_extract:
    print(letterboxd_RDDs[member].first())
    print(letterboxd_RDDs[member].count())

In [ ]:
movies_RDD = letterboxd_RDDs['crew']
for member in members_to_extract[2:]:
    movies_RDD = movies_RDD.join(letterboxd_RDDs[member]).mapValues(lambda x: {**x[0], **x[1]})

In [ ]:
movies_RDD.first()

In [ ]:
letterboxd_RDDs['actors'].count()

In [ ]:
movies_RDD.count()

# Data pre-processing

At the moment our data is contained in RDD form in `movies_RDD`, where each row is a tuple of type `(movie_id, info_list)`. In turn the list of information saved for each movie contains, in order